# PyTorch Fundamentals — Your First Clothing Classifier
**PyTorch Community Meetup Mbarara · 25 minutes**

Goal: train a tiny neural network from scratch to classify clothing pictures. Runtime: CPU is sufficient.

## 1. Tensors are the building blocks (3 minutes)
A tensor is a multidimensional array. Images in Fashion-MNIST are 28 × 28 pixels.

In [ ]:
import torch
print("PyTorch version:",torch.__version__)
pixels = torch.rand(28, 28)
print("Single image shape:", pixels.shape)
print("Mean pixel value:", pixels.mean().item())

## 2. Load clothing images (3 minutes)
Fashion-MNIST has 10 everyday clothing categories.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

transform = transforms.ToTensor()
train_full = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
classes = train_full.classes
# Smaller subset to ensure a CPU-friendly live workshop.
gen = torch.Generator().manual_seed(42)
train_ids = torch.randperm(len(train_full), generator=gen)[:6000].tolist()
train_data = Subset(train_full, train_ids)
test_data = Subset(test_full, list(range(1000)))
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)
print(classes)

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(9, 5))
for ax, img, lab in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(classes[lab.item()])
    ax.axis("off")
plt.tight_layout()

## 3. Define a neural network (4 minutes)
The model flattens each 28×28 image into 784 values and predicts one of 10 clothing classes.

In [ ]:
from torch import nn
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
print(model)

## 4. Train (6 minutes)
Cross-entropy measures prediction mistakes. Backpropagation computes gradients; Adam updates weights.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch():
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x)
    return total_loss / len(train_loader.dataset)

for epoch in range(2):
    print(f"Epoch {epoch+1}: loss = {train_one_epoch():.3f}")

## 5. Evaluate and inspect predictions (4 minutes)
The test subset is kept separate from training.

In [ ]:
model.eval()
correct = 0
with torch.inference_mode():
    for x, y in test_loader:
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
print(f"Accuracy on 1,000 held-out test images: {correct / len(test_data):.1%}")

In [ ]:
x, y = next(iter(test_loader))
with torch.inference_mode():
    pred = model(x).argmax(dim=1)
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x[i].squeeze(), cmap="gray")
    ax.set_title(f"Pred: {classes[pred[i]]}\nTrue: {classes[y[i]]}")
    ax.axis("off")
plt.tight_layout()

## 6. Your turn (last 3 minutes)
- Change the hidden layer from 128 to 64 or 256 units.
- Run one more epoch. Does accuracy improve?
- Look at an incorrect prediction. Why might the model be confused?

**Bridge to workshop 2:** We trained all weights from scratch. What if we could reuse features already learned from millions of pictures?